# GEO Metadata validated and modified process (Gemini-2.0-flash TRUE & Infered has control TRUE)

- ヒトの熱ストレスに関連するRNA-Seqにおいて、Gemini-2.0-flash の判断と自分の判断の結果、解析に使用するに値するデータのメタデータを整理するnotebook
- 整理する際、なぜこのようなデータフレームの処理をするのかについても記載する
- `geofetch`から取得したデータと`iseq`から取得したデータ
- [../GenAI_validate/geo_metadata_genai_validation_true_control.tsv](../GenAI_validate/geo_metadata_genai_validation_true_control.tsv) に分類されたものについてここに記載する

- 2025-05-05: 全てのFASTQデータをダウンロード
- 2025-05-05: GSE124609のRibo-seqは現在の所除いているが、念の為集めといてもいいかもしれない
- 2025-05-09: 念の為、`library_strategy` が `RNA-seq`のものをconcatenateしておく
- 2025-05-15: 全てのFASTQデータをfastpで処理
- 2025-05-21: GSE124609,GSE32060 のRibo-seqのデータはまだ集めていないので注意
- 2025-06-02: 温度条件を追記
- 2026-02-20: [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)にbulk RNA-seq以外のデータの処理を移動 (`library_strategy` が `RNA-seq`のものも含む)

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(100)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera",
    "tax_lineage",
    "bam_md5",
    "fastq_md5",
    "sra_md5"
]

&nbsp;

&nbsp;

&nbsp;

## 1. GSE253874 (SINGLE) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-04-29
- [Histone H3.3 chaperone HIRA renders stress-responsive genes poised for prospective lethal stresses in acquired tolerance](https://doi.org/10.1111/gtc.13140)
- コントロールとなるデータ (37℃ で処理されたサンプル)が存在するため、今回の解析で使用する
- すべてのデータを使用する

#### sample discription from PEP format csv

- `heat stress treatment (40°C, 2 hours)`
- `control`

In [4]:
GSE253874_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE253874/GSE253874_PEP/GSE253874_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq Stranded Total RNA Library Prep Kit").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.1111/gtc.13140").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("40 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE253874 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE253874_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE253874_filtered = GSE253874[[s.name for s in GSE253874 if not (s.null_count() == GSE253874.height)]]


GSE253874_combined = GSE253874_PEP.join(
    GSE253874_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left"
).sort(
    [
        "gsm_id"
    ],
    descending=False
)

GSE253874_combined = GSE253874_combined[[s.name for s in GSE253874_combined if not (s.null_count() == GSE253874_combined.height)]]

GSE253874_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE253874_modified_metadata.tsv",
    separator='\t'
)

# display(GSE253874_combined)

In [5]:
defined_pairs_gse253874 = [
    ("SRR27681984", "SRR27681986"), # heat-stress sample vs control sample
    ("SRR27681983", "SRR27681985") # heat-stress sample vs control sample
]

paired_metadata_df_gse253874 = create_paired_metadata(
    combined_df=GSE253874_combined, 
    defined_sample_pairs=defined_pairs_gse253874,
    sample_id_column_name="accession" # default value, but specified to clarify
)


# Guard against None so that write_csv is called only on a DataFrame
if paired_metadata_df_gse253874 is not None:
    paired_metadata_df_gse253874.write_csv(
        "../../Data/metadata/metadata_paired/GSE253874_paired_metadata.tsv",
        separator="\t"
    )
    display(paired_metadata_df_gse253874)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,mapping,expression_estimation,deg_analysis,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR27681984""","""SRR27681986""","""wi-38-htert_40°c_rna-seq_rep1 vs wi-38-htert_37°c_rna-seq_rep1""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""GSM8029688 vs GSM8029686""","""Public on Oct 15 2024""","""Jan 22 2024""","""Oct 15 2024""","""SRA""","""1""","""WI-38-hTERT""","""Homo sapiens""","""9606""","""total RNA""","""40C_RNA-seq_rep1, 2019_005A_40C_RNA-seq_rep1 vs 37C_RNA-seq_rep1, 2019_005A_37C_RNA-seq_rep1""","""GPL11154""","""Yoshikazu,,Nagagaki""","""Kyoto University""","""Yoshida-Konoe-cho, Sakyo-ku""","""Kyoto-shi""","""Kyoto""","""606-8501""","""Japan""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8029688 vs GSM8029686""","""NONE""","""GSE253874""","""0""","""SRX23349232 vs SRX23349230""","""WI-38-hTERT""","""immortalized human fibroblast cells""","""WT""","""heat stress treatment (40°C, 2 hours) vs control""","""STAR v2.7.0d""","""RSEM v1.3.1""","""edgeR""","""UCSC hg19""","""excel file includes CPM values for each sample""","""SAMN39538492 vs SAMN39538494""","""TruSeq Stranded Total RNA Library Prep Kit""","""https://doi.org/10.1111/gtc.13140""","""40 C""","""120 min.""","""37 C""","""SRR27681984 vs SRR27681986""","""Illumina HiSeq 2000 sequencing: GSM8029688: WI-38-hTERT, 40°C, RNA-seq, rep1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM8029686: WI-38-hTERT, 37°C, RNA-seq, rep1 Homo sapiens RNA-Seq""","""Histone H3.3 chaperone HIRA is required for acquired tolerance by rendering stress-responsive genes poised for prospective lethal stresses [RNA-seq]""","""SRA1789736""","""cDNA""","""2024-10-16""","""670528655 vs 759706171""","""1167544584 vs 1327197055""","""ILLUMINA""","""missing""","""Generic""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""SRP485229""","""36065778 vs 40945648""","""Histone H3.3 chaperone HIRA is required for acquired tolerance by rendering stress-responsive genes poised for prospective lethal stresses [RNA-seq]""","""WI-38-hTERT, 40°C, RNA-seq, rep1 vs WI-38-hTERT, 37°C, RNA-seq, rep1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM8029688: WI-38-hTERT, 40°C, RNA-seq, rep1 Homo sapiens RNA-Seq vs Illumina

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. GSE235351 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-04-29
- [Mechanical confinement promotes heat resistance of hepatocellular carcinoma via SP1/IL4I1/AHR axis](https://doi.org/10.1016/j.xcrm.2023.101128)
- keyword: `Thermal ablation`
- 熱処理の時間はおそらく15分だが、はっきりとした温度 (決められた温度帯か？) がわからなかったのでメールしてみた 
- おそらくgradient に温度を変化させていると思われる
- しかしながら、コントロールっぽいものと熱ストレス曝露条件のデータは見られるので念の為収集
- `M3D_Ctrl_{数字}_RNA_seq` と `M3D_heat_{数字}_RNA_seq` のサンプルを収集

In [6]:
regex_pattern_gse235351 = r'^m3d_(ctrl|heat)_\d+_rna_seq$' # Remove double quotes from the pattern

GSE235351_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE235351/GSE235351_PEP/GSE235351_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (Lianchuan Biotechnology Co., Ltd)
    pl.lit("https://doi.org/10.1016/j.xcrm.2023.101128").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("37~53 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("15 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
).filter(
    # Filter using the regex pattern without quotes
    pl.col("sample_name").str.contains(regex_pattern_gse235351) 
)

# metadata from iseq
GSE235351 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE235351_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE235351_filtered = GSE235351[[s.name for s in GSE235351 if not (s.null_count() == GSE235351.height)]]

GSE235351_combined = GSE235351_PEP.join(
    GSE235351_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE235351_combined = GSE235351_combined[[s.name for s in GSE235351_combined if not (s.null_count() == GSE235351_combined.height)]]

GSE235351_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE235351_modified_metadata.tsv",
    separator='\t'
)

# display(GSE235351_combined)

In [7]:
# Manual defined pairs
defined_pairs_gse235351 = [
    ("SRR24973772", "SRR24973776"), # heat-stress sample vs control sample
    ("SRR24973774", "SRR24973775"), # heat-stress sample vs control sample
    ("SRR24973771", "SRR24973773") # heat-stress sample vs control sample
]

paired_metadata_df_gse235351 = create_paired_metadata(
    combined_df=GSE235351_combined, 
    defined_sample_pairs=defined_pairs_gse235351,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse235351 is not None:
    paired_metadata_df_gse235351.write_csv(
        "../../Data/metadata/metadata_paired/GSE235351_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse235351)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_data_processing,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,biological_replicates,treatment,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR24973772""","""SRR24973776""","""m3d_heat_1_rna_seq vs m3d_ctrl_1_rna_seq""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""GSM7500881 vs GSM7500878""","""Public on Jun 25 2023""","""Jun 20 2023""","""Jun 25 2023""","""SRA""","""1""","""Hepatocellular carcinoma""","""Homo sapiens""","""9606""","""total RNA""","""HISAT2""","""GPL30209""","""Guopei,,Zhang""","""zhanggp6@mail2.sysu.edu.cn""","""Sun Yat-sen University""","""58 Zhongshan Road 2""","""Guangzhou""","""510080""","""China""","""MGISEQ-2000RS""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7500881 vs GSM7500878""","""NONE""","""GSE235351""","""0""","""SRX20731424 vs SRX20731421""","""Hepatocellular carcinoma""","""MHCC-97H""","""Hepatocellular Carcinoma Cell Line""","""biological replicate_1""","""MHCC-97H in the 3D model-heat treatment vs MHCC-97H in the 3D model-treatment control""","""hg19""","""text file includes raw counts for each Sample""","""SAMN35809839 vs SAMN35809842""","""unknown""","""https://doi.org/10.1016/j.xcrm.2023.101128""","""37~53 C""","""15 min.""","""37 C (inferred)""","""SRR24973772 vs SRR24973776""","""MGISEQ-2000RS sequencing: GSM7500881: M3D_Heat_1_RNA_seq Homo sapiens RNA-Seq vs MGISEQ-2000RS sequencing: GSM7500878: M3D_Ctrl_1_RNA_seq Homo sapiens RNA-Seq""","""Mechanical Confinement Promotes Heat Resistance of Hepatocellular Carcinoma via SP1/IL4I1/AHR Axis""","""SRA1658946""","""cDNA""","""2023-06-27""","""653569416 vs 648134640""","""582748331 vs 588809244""","""BGISEQ""","""missing""","""Generic""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""SRP444919""","""23705124 vs 23832919""","""Mechanical Confinement Promotes Heat Resistance of Hepatocellular Carcinoma via SP1/IL4I1/AHR Axis""","""M3D_Heat_1_RNA_seq vs M3D_Ctrl_1_RNA_seq""","""MGISEQ-2000RS""","""MGISEQ-2000RS sequencing: GSM7500881: M3D_Heat_1_RNA_seq Homo sapiens RNA-Seq vs MGISEQ-2000RS sequencing: GSM7500878: M3D_Ctrl_1_RNA_seq Homo sapiens RNA-Seq""","""Hepatocellular Carcinoma Cell Line""","""1185256200 vs 1191645950""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM7500881 vs GSM7500878""","""GSM7500881_r1 vs GSM7500878_r1""","""RNA of the 2D, 3D, and animal models were extracted using Rneasy Min

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE165367 (SINGLE) 4sU-seq 

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (4sU-RNA標識法)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE165364 (SINGLE) TTchem-seq 

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (TTchem-seq)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- しかしながら、nacsent RNA のバイアスを考慮できているかが気になるので後でまた使用するかどうか検討する
- [Using TTchem-seq for profiling nascent transcription and measuring transcript elongation](https://doi.org/10.1038/s41596-019-0262-3) などの論文を見ると、STARのマッピングなどを使用しているため、おそらく発現定量の方法はRNA-seqと同じ？
- 120分の実験を行ったあと、追加で30,60,120min.の実験をしている
- 異なるペアで熱ストレスの条件が異なるので後で手入力
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE165366 (SINGLE) FP-TTchem-seq 

- validate date 2025-04-29
- [Heat shock induces premature transcript termination and reconfigures the human transcriptome](https://doi.org/10.1016/j.molcel.2022.01.007)
- 以前の研究でも使用したデータ (FP:Flavopiridol.TTchem-seq)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- しかしながら、nacsent RNA のバイアスを考慮できているかがきになるので後でまた使用するかどうか検討する
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

### GSE210563 (SINGLE) NO DOWNLOAD

- validate date 2025-04-29
- 熱ショックを加えているサンプルがヒトではなくマウスのMEFなのでこれは使用しない

#### sample discription from PEP format csv

- `MEF_heatshock-_GLORI+_rep1`
- `MEF_heatshock-_GLORI-,`
- ...

In [8]:
# GSE210563_PEP = pl.read_csv(
#     "../Data/metadata/metadata_geofetch/GSE210563/GSE210563_PEP/GSE210563_PEP_raw.csv",
#     separator=','
# ).filter(
#     pl.col("sample_name").str.contains(r'heatshock')
# )

# # display(GSE210563_PEP)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 3. GSE124609 (SINGLE) RNA-seq

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date 2025-04-29
- [Cellular proteostasis decline in human senescence](https://doi.org/10.1073/pnas.2018138117)
- コントロールと熱ストレス条件のデータが存在するため、すべてのデータを使用する予定...
- Ribo-seqのデータの処理を[./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に移動

#### sample discription from PEP format csv

- `Young1_RNA_Seq`
- `Young_HS1_RNA_Seq`
- ...

In [9]:
GSE124609_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE124609/GSE124609_PEP/GSE124609_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina kit (TruSeq RNA)").alias("rna_library"), # add rna_library column (Illumina: from Article)
    pl.lit("https://doi.org/10.1073/pnas.2018138117").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("44 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# rna-seq
GSE124609_PEP_rna_seq = GSE124609_PEP.filter(
    pl.col("sample_name").str.contains(r'rna_seq')
)


# metadata from iseq
GSE124609 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE124609_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE124609_filtered = GSE124609[[s.name for s in GSE124609 if not (s.null_count() == GSE124609.height)]]

GSE124609_rna_seq_combined = GSE124609_PEP_rna_seq.join(
    GSE124609_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE124609_rna_seq_combined = GSE124609_rna_seq_combined[[s.name for s in GSE124609_rna_seq_combined if not (s.null_count() == GSE124609_rna_seq_combined.height)]]

GSE124609_rna_seq_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE124609_rna_seq_modified_metadata.tsv",
    separator='\t'
)

# display(GSE124609_rna_seq_combined)

In [10]:
# Manual defined pairs
defined_pairs_gse124609_rna_seq = [
    ("SRR8392699", "SRR8392697"),
    ("SRR8392700", "SRR8392698"),
    ("SRR8392703", "SRR8392701"),
    ("SRR8392704", "SRR8392702")
]

paired_metadata_df_gse124609_rna_seq = create_paired_metadata(
    combined_df=GSE124609_rna_seq_combined, 
    defined_sample_pairs=defined_pairs_gse124609_rna_seq,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse124609_rna_seq is not None:
    paired_metadata_df_gse124609_rna_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE124609_paired_rna_seq_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse124609_rna_seq)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,passage,phenotype,treatment,replicate,genome_build,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR8392699""","""SRR8392697""","""young_hs1_rna_seq vs young1_rna_seq""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR8392699 vs SRR8392697""","""SRX5202410 vs SRX5202408""","""Young_HS1_RNA_Seq vs Young1_RNA_Seq""","""GSM3537584 vs GSM3537582""","""Public on Dec 09 2020""","""Jan 03 2019""","""Dec 09 2020""","""SRA""","""1""","""WI38 cells""","""Homo sapiens""","""9606""","""total RNA""","""GPL11154""","""Reut,,Shalgi""","""reutshalgi@technion.ac.il""","""Faculty of Medicine""","""Technion""","""1 Efron st.""","""Haifa""","""31096""","""Israel""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3537584 vs GSM3537582""","""NONE""","""GSE124609""","""0""","""SRX5202410 vs SRX5202408""","""WI38""","""human lung fibroblasts""","""24""","""young""","""Heat shock vs Control""","""1""","""hg19""","""Samples read counts and TPM values""","""SAMN10689181 vs SAMN10689183""","""Illumina kit (TruSeq RNA)""","""https://doi.org/10.1073/pnas.2018138117""","""44 C""","""120 min.""","""37 C (inferred)""","""SRR8392699 vs SRR8392697""","""Illumina HiSeq 2000 sequencing: GSM3537584: Young_HS1_RNA_Seq Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3537582: Young1_RNA_Seq Homo sapiens RNA-Seq""","""Cellular proteostasis collapse in mammalian senescence""","""SRA830119""","""cDNA""","""2021-01-04""","""668114167 vs 709782808""","""1112311397 vs 1181685001""","""ILLUMINA""","""Generic""","""Young_HS1_RNA_Seq vs Young1_RNA_Seq""","""SRP175158""","""39578011 vs 41968749""","""Cellular proteostasis collapse in mammalian senescence""","""Young_HS1_RNA_Seq vs Young1_RNA_Seq""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM3537584: Young_HS1_RNA_Seq Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM3537582: Young1_RNA_Seq Homo sapiens RNA-Seq""","""human lung fibroblasts""","""2018478561 vs 2140406199""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM3537584 vs GSM3537582""","""GSM3537584_r1 vs GSM3537582_r1""","""RNA-Seq libraries: RNA was isolated from collected samples using the TRIZOL reagent (Thermo) following treatment with Turbo DNase and phenol chloroform precipitation. RN

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE140053 (SINGLE) SLAM-seq

- validation date: 2025-04-29
- [Stress-induced nuclear condensation of NELF drives transcriptional downregulation](https://doi.org/10.1016/j.molcel.2021.01.016)
- これも以前の研究で使用したデータ (SLAM-seq)
- 定量できるかわからないが、これも集めよう...
- ChiP-seqのデータは取り除く ([./12_8_metadata_chipseq.ipynb](./12_8_metadata_chipseq.ipynb)に処理を移動)
- `hs_nelfa_didr_rep3` のデータがないので、ペアとなる`nhs_nelfa_wt_rep3` は今回は使用しない
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 4. GSE115973 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validate date: 2025-04-29
- [Heat Shock Factor 2 Protects against Proteotoxicity by Maintaining Cell-Cell Adhesion](https://doi.org/10.1016/j.celrep.2019.12.037)
- これも以前の研究で使用したデータ
- heat shock 以外の処理も含まれているため、これらを取り除く
- HSF1,2 KO データ含む
- GEOのIDに対して、srrのIDが複数 (`srr`カラムが `multi`) なので対応するように気をつける

In [11]:
GSE115973_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE115973/GSE115973_PEP/GSE115973_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("treatment").str.contains(r'Bortezomib')) & # remove only Bortezomib treatment data
    ((pl.col("treatment").str.contains(r'control')) |
    (pl.col("treatment").str.contains(r'heat shock')))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Illumina TruSeq Stranded mRNA Sample Preparation Guide").alias("rna_library"), # add rna_library column (Lexogen)
    pl.lit("https://doi.org/10.1016/j.celrep.2019.12.037").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE115973 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE115973_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE115973_filtered = GSE115973[[s.name for s in GSE115973 if not (s.null_count() == GSE115973.height)]]

GSE115973_combined = GSE115973_PEP.join(
    GSE115973_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
).sort(
    "sample_geo_accession", descending=True
)

GSE115973_combined = GSE115973_combined[[s.name for s in GSE115973_combined if not (s.null_count() == GSE115973_combined.height)]]

GSE115973_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE115973_modified_metadata.tsv",
    separator='\t'
)

# display(GSE115973_PEP)

In [12]:
# Manual defined pairs
defined_pairs_gse115973 = [
    ("SRR7359799", "SRR7359790"),
    ("SRR7359798", "SRR7359789"),
    ("SRR7359797", "SRR7359788"),
    ("SRR7359787", "SRR7359778"),
    ("SRR7359786", "SRR7359777"),
    ("SRR7359785", "SRR7359776"),
    ("SRR7359775", "SRR7359766"),
    ("SRR7359774", "SRR7359765"),
    ("SRR7359773", "SRR7359764"),
    ("SRR7359763", "SRR7359754"),
    ("SRR7359762", "SRR7359753"),
    ("SRR7359761", "SRR7359752"),
    ("SRR7359751", "SRR7359742"),
    ("SRR7359750", "SRR7359741"),
    ("SRR7359749", "SRR7359740"),
    ("SRR7359739", "SRR7359730"),
    ("SRR7359738", "SRR7359729"),
    ("SRR7359737", "SRR7359728"),
    ("SRR7359727", "SRR7359718"),
    ("SRR7359726", "SRR7359717"),
    ("SRR7359725", "SRR7359716"), # 2026-01-17: SRR7359725 を発現定量し忘れていた
    ("SRR7359715", "SRR7359706"),
    ("SRR7359714", "SRR7359705"),
    ("SRR7359713", "SRR7359704"),
    ("SRR7359703", "SRR7359694"),
    ("SRR7359702", "SRR7359693"),
    ("SRR7359701", "SRR7359692"),
    ("SRR7359691", "SRR7359682"),
    ("SRR7359690", "SRR7359681"),
    ("SRR7359689", "SRR7359680"),
    ("SRR7359679", "SRR7359670"),
    ("SRR7359678", "SRR7359669"),
    ("SRR7359677", "SRR7359668"),
    ("SRR7359667", "SRR7359658"),
    ("SRR7359666", "SRR7359657"),
    ("SRR7359665", "SRR7359656")
]

paired_metadata_df_gse115973 = create_paired_metadata(
    combined_df=GSE115973_combined, 
    defined_sample_pairs=defined_pairs_gse115973,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse115973 is not None:
    paired_metadata_df_gse115973.write_csv(
        "../../Data/metadata/metadata_paired/GSE115973_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse115973)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,sample_supplementary_file_3,gse_id,sample_data_row_count,experiment_accession,cell_line,disease,genotype,treatment,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR7359799""","""SRR7359790""","""hs_2ko_rep4 vs c_2ko_rep4""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""multi""","""SRX4232775 vs SRX4232772""","""HS_2KO_rep4 vs C_2KO_rep4""","""GSM3197068 vs GSM3197065""","""Public on Jan 14 2020""","""Jun 18 2018""","""Jan 15 2020""","""SRA""","""1""","""cell""","""Homo sapiens""","""9606""","""total RNA""","""sequencing run made in 3 lanes""","""GPL21290""","""Jenny,Maaria,Joutsen""","""jenny.joutsen@abo.fi""","""Lea Sistonen""","""Cell Biology""","""Åbo Akademi University""","""Tykistökatu 6A""","""Turku""","""Varsinais-Suomi""","""20520""","""Finland""","""Illumina HiSeq 3000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3197068 vs GSM3197065""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197068/suppl/GSM3197068_48_HS_2KO_S48_L001.txt.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197065/suppl/GSM3197065_45_C_2KO_S30_L0…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197068/suppl/GSM3197068_48_HS_2KO_S48_L002.txt.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197065/suppl/GSM3197065_45_C_2KO_S30_L0…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197068/suppl/GSM3197068_48_HS_2KO_S48_L005.txt.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM3197nnn/GSM3197065/suppl/GSM3197065_45_C_2KO_S30_L0…","""GSE115973""","""0""","""SRX4232775 vs SRX4232772""","""U2OS""","""osteosarcoma""","""heat shock farctor 2 KO""","""heat shock 1 h in 42 degrees vs control""","""human hg38""","""SAMN09445400 vs SAMN09445404""","""Illumina TruSeq Stranded mRNA Sample Preparation Guide""","""https://doi.org/10.1016/j.celrep.2019.12.037""","""42 C""","""60 min.""","""37 C (inferred)""","""SRR7359799 vs SRR7359790""","""Illumina HiSeq 3000 sequencing: GSM3197068: HS_2KO_rep4 Homo sapiens RNA-Seq vs Illumina HiSeq 3000 sequencing: GSM3197065: C_2KO_rep4 Homo sapiens RNA-Seq""","""Impact of HSF1 and HSF2 on gene expression in normal growth conditions and upon proteotoxic stress.""","""SRA724426""","""cDNA""","""2020-01-16""","""145038492 vs 141234067""","""216033479 vs 210439125""","""ILLUMINA""","""Generic"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 5. GSE79680 (SINGLE)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-04-29
- [RNA Directed Modulation of Phenotypic Plasticity in Human Cells](https://doi.org/10.1371/journal.pone.0152424)
- lncRNAについて調べている論文
- しっかりコントロールと熱ストレス条件のデータが揃っていたので収集
- 45℃, 30分

In [13]:
GSE79680_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE79680/GSE79680_PEP/GSE79680_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("Truseq RNA prep kit").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1371/journal.pone.0152424").alias("reference_doi"), # add DOI of the reference paper column
    pl.lit("45 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE79680 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE79680_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE79680_filtered = GSE79680[[s.name for s in GSE79680 if not (s.null_count() == GSE79680.height)]]

GSE79680_combined = GSE79680_PEP.join(
    GSE79680_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE79680_combined = GSE79680_combined[[s.name for s in GSE79680_combined if not (s.null_count() == GSE79680_combined.height)]]

GSE79680_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE79680_modified_metadata.tsv",
    separator='\t'
)

# display(GSE79680_combined)

In [14]:
# Manual defined pairs
defined_pairs_gse79680 = [
    ("SRR3309259", "SRR3309256"),
    ("SRR3309260", "SRR3309257"),
    ("SRR3309261", "SRR3309258")
]

paired_metadata_df_gse79680 = create_paired_metadata(
    combined_df=GSE79680_combined, 
    defined_sample_pairs=defined_pairs_gse79680,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse79680 is not None:
    paired_metadata_df_gse79680.write_csv(
        "../../Data/metadata/metadata_paired/GSE79680_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse79680)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,stress,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR3309259""","""SRR3309256""","""heatshock1 vs control1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR3309259 vs SRR3309256""","""SRX1668367 vs SRX1668364""","""HeatShock1 vs Control1""","""GSM2101093 vs GSM2101090""","""Public on Mar 30 2016""","""Mar 29 2016""","""May 15 2019""","""SRA""","""1""","""HeatShock vs Control""","""Homo sapiens""","""9606""","""total RNA""","""rRNA depleted total RNA""","""GPL11154""","""Jon,,Burdach""","""BABS""","""University of New South Wales""","""UNSW""","""Sydney""","""NSW""","""2052""","""Australia""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM2101093 vs GSM2101090""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2101nnn/GSM2101093/suppl/GSM2101093_HS1.fpkm_tracking.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2101nnn/GSM2101090/suppl/GSM2101090_Control1.fpkm_tracki…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2101nnn/GSM2101093/suppl/GSM2101093_HS1.gtf.gz vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM2101nnn/GSM2101090/suppl/GSM2101090_Control1.gtf.gz""","""GSE79680""","""0""","""SRX1668367 vs SRX1668364""","""HEK293 cells""","""heat shock vs control""","""hg19""","""SAMN04590838 vs SAMN04590835""","""Truseq RNA prep kit""","""https://doi.org/10.1371/journal.pone.0152424""","""45 C""","""30 min.""","""37 C (inferred)""","""SRR3309259 vs SRR3309256""","""Illumina HiSeq 2000 sequencing: GSM2101093: HeatShock1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM2101090: Control1 Homo sapiens RNA-Seq""","""Homo sapiens""","""SRA399583""","""cDNA""","""2019-12-22 vs 2019-10-30""","""941131424 vs 1550775840""","""1157288567 vs 1936682237""","""ILLUMINA""","""Generic""","""HeatShock1 vs Control1""","""SRP072687""","""12905137 vs 21237711""","""HEK293 Heat-shock experiment""","""HeatShock1 vs Control1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM2101093: HeatShock1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM2101090: Control1 Homo sapiens RNA-Seq""","""1445375344 vs 2378623632""","""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM2101093 vs GSM2101090""","""GSM2101093_r1 vs GSM2101090_r1""","""RNA was extracted using Trizol according to the manufacturer's instructions

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE32060 (SINGLE) Ribo-seq

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [Widespread Regulation of Translation by Elongation Pausing in Heat Shock](https://doi.org/10.1016/j.molcel.2012.11.028)
- 以前の研究でも使用した研究
- ペアごとに熱ストレスの条件に違うので、後で手入力
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 6. GSE267930 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date 2025-04-29
- [Inflammatory stress-mediated chromatin changes underlie dysfunction in endothelial cells](https://doi.org/10.1101/2023.10.11.561959)
- BioRxivのデータ
- データとしてはすべてRNA-seqのデータで TNF-alpha 処理のデータも含まれていたのでこちらを除外する
- 収集できるデータとして使用 (39℃, 2h.)

In [15]:
GSE267930_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE267930/GSE267930_PEP/GSE267930_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("sample_title").str.contains(r'TNF-alpha')) # remove TNF-alpha treatment data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra II Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column (Illumina)
    pl.lit("https://doi.org/10.1101/2023.10.11.561959").alias("reference_doi"), # add DOI of the reference paper column (BioRxiv)
    pl.lit("39 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("120 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE267930 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE267930_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE267930_filtered = GSE267930[[s.name for s in GSE267930 if not (s.null_count() == GSE267930.height)]]

GSE267930_combined = GSE267930_PEP.join(
    GSE267930_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE267930_combined = GSE267930_combined[[s.name for s in GSE267930_combined if not (s.null_count() == GSE267930_combined.height)]]

GSE267930_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE267930_modified_metadata.tsv",
    separator='\t'
)

# display(GSE267930_combined)

In [16]:
# Manual defined pairs
defined_pairs_gse267930 = [
    ("SRR29089688", "SRR29089691"), # heat shock vs control
    ("SRR29089687", "SRR29089690"),
    ("SRR29089686", "SRR29089689"),
]

paired_metadata_df_gse267930 = create_paired_metadata(
    combined_df=GSE267930_combined, 
    defined_sample_pairs=defined_pairs_gse267930,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse267930 is not None:
    paired_metadata_df_gse267930.write_csv(
        "../../Data/metadata/metadata_paired/GSE267930_modified_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse267930)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_phone,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,treatment,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR29089688""","""SRR29089691""","""huvec_39_°c_heat_stress_for_2_hours_biological_replicate_1_hs1_ vs huvec_control_biological_replicate_1_c1_""","""HUVEC, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, control, biological replicate 1 (C1)""","""GSM8281737 vs GSM8281734""","""Public on May 25 2024""","""May 20 2024""","""May 25 2024""","""SRA""","""1""","""HUVEC""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Haibo,,Liu""","""haibo.liu@umassmed.edu""","""5153573719""","""MCCB""","""UMass Chan""","""164 Plantation St""","""Unionville""","""CT""","""6085""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8281737 vs GSM8281734""","""NONE""","""GSE267930""","""0""","""SRX24613831 vs SRX24613828""","""HUVEC""","""vascular endothelial""","""heat stress vs control""","""Tab-delimited (.tsv) gene-by-sample count table""","""SAMN41463628 vs SAMN41463631""","""NEBNext Ultra II Directional RNA Library Prep Kit""","""https://doi.org/10.1101/2023.10.11.561959""","""39 C""","""120 min.""","""37 C (inferred)""","""SRR29089688 vs SRR29089691""","""Illumina NovaSeq 6000 sequencing: GSM8281737: HUVEC, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8281734: HUVEC, control, b…","""Inflammatory stress-mediated chromatin changes underlie dysfunction in endothelial cells. [RNA-Seq]""","""SRA1871897""","""cDNA""","""2024-06-30""","""2086890183 vs 2273963587""","""1521126806;1539877789 vs 1654146828;1691572272""","""ILLUMINA""","""missing""","""Generic""","""HUVEC, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, control, biological replicate 1 (C1)""","""SRP508793""","""23334827 vs 25026624""","""Inflammatory stress-mediated chromatin changes underlie dysfunction in endothelial cells. [RNA-Seq]""","""HUVEC, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) vs HUVEC, control, biological replicate 1 (C1)""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM8281737: HUVEC, 39 °C heat stress for 2 hours, biological replicate 1 (HS1) Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8281734: HUVEC, control, b…",

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 7. GSE271918 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-04-29
- [Effect of CLK2 expression in the regulation of intron retention during heat shock](https://doi.org/10.1002/advs.202309588)
- CLK2のノックダウンなどが行われている研究
- ~~`T343A-N`, `T343A-C` などのmutationを導入したサンプルもあるが、これもコントロールとして使用する予定 (ペア数の増加に寄与)~~
- 42℃, 2h
- **mutationを加えているT343A-C, T343A-Nのデータに対して熱ストレスを加えているデータは含まれていないので今回は使用しない**
- **`HeLa cells, Control`に対して、`HeLa cells,HS-Control`, `HeLa cells,HS-WT`, `HeLa cells,HS-KD`** の3種類のペアを作成する

In [17]:
GSE271918_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE271918/GSE271918_PEP/GSE271918_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("sample_title").str.contains(r'T343A-')) # remove T343A-N, T343A-C mutation data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (from supproting information) "The RNA library construction and sequencing were performed by Bioyi Biotechnology Co., Ltd. Wuhan, China (details can be found in the Supporting Information). "
    pl.lit("https://doi.org/10.1002/advs.202309588").alias("reference_doi"), # add DOI of the reference paper column (BioRxiv)
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min.").alias("stress treatment time"), # add heat treatment time column (minutes) (from supporting information Fig. S3D)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE271918 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE271918_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE271918_filtered = GSE271918[[s.name for s in GSE271918 if not (s.null_count() == GSE271918.height)]]

GSE271918_combined = GSE271918_PEP.join(
    GSE271918_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE271918_combined = GSE271918_combined[[s.name for s in GSE271918_combined if not (s.null_count() == GSE271918_combined.height)]]

GSE271918_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE271918_modified_metadata.tsv",
    separator='\t'
)

# display(GSE271918_combined)

In [18]:
# Manual defined pairs
defined_pairs_gse271918 = [
    ("SRR29765556", "SRR29765559"), # heat shock vs control
    ("SRR29765555", "SRR29765558"),
    ("SRR29765554", "SRR29765557"),
    ("SRR29765553", "SRR29765559"),
    ("SRR29765552", "SRR29765558"),
    ("SRR29765551", "SRR29765557"),
    ("SRR29765550", "SRR29765559"),
    ("SRR29765549", "SRR29765558"),
    ("SRR29765548", "SRR29765557")
]

paired_metadata_df_gse271918 = create_paired_metadata(
    combined_df=GSE271918_combined, 
    defined_sample_pairs=defined_pairs_gse271918,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse271918 is not None:
    paired_metadata_df_gse271918.write_csv(
        "../../Data/metadata/metadata_paired/GSE271918_modified_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse271918)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR29765556""","""SRR29765559""","""hela_cells_hs-control-1 vs hela_cells_control-1""","""HeLa cells,HS-Control-1 vs HeLa cells, Control-1""","""GSM8389003 vs GSM8389000""","""Public on Jul 15 2024""","""Jul 10 2024""","""Jul 15 2024""","""SRA""","""1""","""HeLa""","""Homo sapiens""","""9606""","""total RNA""","""HeLa-HS1 vs HeLa1""","""GPL29480""","""Wei,,Wang""","""wekle@hust.edu.cn""","""13 Hangkong Road""","""Wuhan""","""Hubei""","""430030""","""China""","""DNBSEQ-T7""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8389003 vs GSM8389000""","""NONE""","""GSE271918""","""0""","""SRX25266181 vs SRX25266178""","""HeLa""","""cervical cancer cell line""","""Control""","""Heat shock vs Untreated""","""GRCh38.105""","""tab-delimited text files include raw counts""","""SAMN42396141 vs SAMN42396144""","""unknown""","""https://doi.org/10.1002/advs.202309588""","""42 C""","""240 min.""","""37 C (inferred)""","""SRR29765556 vs SRR29765559""","""DNBSEQ-T7 sequencing: GSM8389003: HeLa cells,HS-Control-1 Homo sapiens RNA-Seq vs DNBSEQ-T7 sequencing: GSM8389000: HeLa cells, Control-1 Homo sapiens RNA-Seq""","""Effect of CLK2 expression in the regulation of intron retention during heat shock [RNA-seq]""","""SRA1921142""","""cDNA""","""2024-07-29""","""7415990718 vs 5505199772""","""4410725838;4854233767 vs 3433477831;3490574318""","""DNBSEQ""","""missing""","""Generic""","""HeLa cells,HS-Control-1 vs HeLa cells, Control-1""","""SRP519108""","""45949798 vs 34301509""","""Effect of CLK2 expression in the regulation of intron retention during heat shock [RNA-seq]""","""HeLa cells,HS-Control-1 vs HeLa cells, Control-1""","""DNBSEQ-T7""","""DNBSEQ-T7 sequencing: GSM8389003: HeLa cells,HS-Control-1 Homo sapiens RNA-Seq vs DNBSEQ-T7 sequencing: GSM8389000: HeLa cells, Control-1 Homo sapiens RNA-Seq""","""cervical cancer cell line""","""13784939400 vs 10290452700""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM8389003 vs GSM8389000""","""GSM8389003_r1 vs GSM8389000_r1""","""RNAs were extracted using Trizol. mRNA isolation from total RNA was executed utilizing poly-T oligo-attached magnetic beads. Subsequently, mRNA underwent fragmentation in a buffer containing divalent …","""missing""","""GSM8389003_r1 vs GSM8389000_r1"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 8. GSE259306 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-04-29
- [TurboCas: A method for locus-specific labeling of genomic regions and isolating their associated protein interactome](https://doi.org/10.1016/j.molcel.2024.11.007)
- Hspa6,などのKnockdownを行った際の熱ストレス応答について研究している論文とデータ
- Turbocas というツールが主眼
- 42℃, 3h
- コントロール、熱ストレス曝露のデータが揃っているため、解析に使用する
- `neg`が `37C 37 hours, 3h`, `pos`が `42C 3 hours (heat shock)`
- 一部のデータが `neg`のデータが一つしかなかったので、`replicate 2`にも`neg`の`replicate 1`に対応させる

In [19]:
GSE259306_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE259306/GSE259306_PEP/GSE259306_PEP_raw.csv",
    separator=','
).with_columns(
    (
        pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
        pl.lit("NEBNext Ultra II Directional RNA Library Prep Kit for Illumina").alias("rna_library"), # add rna_library column (NEB, E7760L)
        pl.lit("https://doi.org/10.1016/j.molcel.2024.11.007").alias("reference_doi"), # add DOI of the reference paper column (BioRxiv)
        pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
        pl.lit("180 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
        pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE259306 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE259306_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE259306_filtered = GSE259306[[s.name for s in GSE259306 if not (s.null_count() == GSE259306.height)]]

GSE259306_combined = GSE259306_PEP.join(
    GSE259306_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE259306_combined = GSE259306_combined[[s.name for s in GSE259306_combined if not (s.null_count() == GSE259306_combined.height)]]

GSE259306_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE259306_modified_metadata.tsv",
    separator='\t'
)

# display(GSE259306_combined)

In [20]:
# Manual defined pairs
defined_pairs_gse259306 = [
    ("SRR28110418", "SRR28110420"), # heat shock vs control (Abcf2 replicate 1)
    ("SRR28110417", "SRR28110419"), # heat shock vs control (Abcf2 replicate 2)
    ("SRR28110414", "SRR28110416"), # heat shock vs control (Acaca replicate 1)
    ("SRR28110413", "SRR28110415"), # heat shock vs control (Acaca replicate 2)
    ("SRR28110402", "SRR28110411"), # heat shock vs control (Acacb replicate 1)
    ("SRR28110403", "SRR28110412"), # heat shock vs control (Acacb replicate 2)
    ("SRR28110406", "SRR28110404"), # heat shock vs control (Amot replicate 1)
    ("SRR28110407", "SRR28110405"), # heat shock vs control (Amot replicate 2)
    ("SRR28110410", "SRR28110408"), # heat shock vs control (Atad3a replicate 1)
    ("SRR28110401", "SRR28110409"), # heat shock vs control (Atad3a replicate 2)
    ("SRR28110400", "SRR28110392"), # heat shock vs control (Bag2 replicate 1)
    ("SRR28110399", "SRR28110393"), # heat shock vs control (Bag2 replicate 2)
    ("SRR28110386", "SRR28110395"), # heat shock vs control (Fubp3 replicate 1)
    ("SRR28110385", "SRR28110394"), # heat shock vs control (Fubp3 replicate 2)
    ("SRR28110391", "SRR28110384"), # heat shock vs control (Hsp90aa1 replicate 1)
    ("SRR28110390", "SRR28110383"), # heat shock vs control (Hsp90aa1 replicate 2)
    ("SRR28110387", "SRR28110389"), # heat shock vs control (Hsp90a1b replicate 1)
    ("SRR28110357", "SRR28110388"), # heat shock vs control (Hsp90a1b replicate 2)
    ("SRR28110354", "SRR28110356"), # heat shock vs control (Hspa6 replicate 1)
    ("SRR28110382", "SRR28110355"), # heat shock vs control (Hspa6 replicate 2)
    ("SRR28110345", "SRR28110358"), # heat shock vs control (Hspa9 replicate 1)
    ("SRR28110344", "SRR28110346"), # heat shock vs control (Hspa9 replicate 2)
    ("SRR28110352", "SRR28110343"), # heat shock vs control (Kpna2 replicate 1)
    ("SRR28110351", "SRR28110353"), # heat shock vs control (Kpna2 replicate 2)
    ("SRR28110348", "SRR28110350"), # heat shock vs control (Nat10 replicate 1)
    ("SRR28110347", "SRR28110349"), # heat shock vs control (Nat10 replicate 2)
    ("SRR28110342", "SRR28110323"), # heat shock vs control (Nkap replicate 1)
    ("SRR28110341", "SRR28110322"), # heat shock vs control (Nkap replicate 2)
    ("SRR28110327", "SRR28110329"), # heat shock vs control (Nono replicate 1)
    ("SRR28110326", "SRR28110328"), # heat shock vs control (Nono replicate 2)
    ("SRR28110317", "SRR28110319"), # heat shock vs control (Rad50 replicate 1)
    ("SRR28110316", "SRR28110318"), # heat shock vs control (Rad50 replicate 2)
    ("SRR28110377", "SRR28110315"), # heat shock vs control (Ranbp2 replicate 1)
    ("SRR28110376", "SRR28110378"), # heat shock vs control (Ranbp2 replicate 2)
    ("SRR28110311", "SRR28110375"), # heat shock vs control (Scrambled replicate 1)
    ("SRR28110310", "SRR28110312"), # heat shock vs control (Scrambled replicate 2)
    ("SRR28110369", "SRR28110380"), # heat shock vs control (Sdad1 replicate 1)
    ("SRR28110368", "SRR28110379"), # heat shock vs control (Sdad1 replicate 2)
    ("SRR28110374", "SRR28110367"), # heat shock vs control (Smarcd2 replicate 1)
    ("SRR28110373", "SRR28110366"), # heat shock vs control (Smarcd2 replicate 2)
    ("SRR28110370", "SRR28110372"), # heat shock vs control (Sumo2 replicate 1)
    ("SRR28110340", "SRR28110371"), # heat shock vs control (Sumo2 replicate 2)
    ("SRR28110363", "SRR28110365"), # heat shock vs control (Trim28 replicate 1)
    ("SRR28110362", "SRR28110364"), # heat shock vs control (Trim28 replicate 2)
    ("SRR28110332", "SRR28110361"), # heat shock vs control (Yy1 replicate 1)
    ("SRR28110331", "SRR28110333"), # heat shock vs control (Yy1 replicate 2)
    ("SRR28110335", "SRR28110330"), # heat shock vs control (Zfp91 replicate 1)
    ("SRR28110334", "SRR28110336"), # heat shock vs control (Zfp91 replicate 2)
    ("SRR28110397", "SRR28110398"), # heat shock vs control (Eef1g replicate 1) (negative data is only SRR28110398)
    ("SRR28110396", "SRR28110398"), # heat shock vs control (Eef1g replicate 2) (negative data is only SRR28110398)
    ("SRR28110360", "SRR28110381"), # heat shock vs control (Hspa8 replicate 1) (negative data is only SRR28110381)
    ("SRR28110359", "SRR28110381"), # heat shock vs control (Hspa8 replicate 2) (negative data is only SRR28110381)
    ("SRR28110324", "SRR28110325"), # heat shock vs control (Prkdc replicate 1) (negative data is only SRR28110325)
    ("SRR28110314", "SRR28110325"), # heat shock vs control (Prkdc replicate 2) (negative data is only SRR28110325)
    ("SRR28110321", "SRR28110313"), # heat shock vs control (Prpf40q replicate 1) (negative data is only SRR28110313)
    ("SRR28110320", "SRR28110313"), # heat shock vs control (Prpf40q replicate 2) (negative data is only SRR28110313)
    ("SRR28110338", "SRR28110339"), # heat shock vs control (Supt16h replicate 1) (negative data is only SRR28110339)
    ("SRR28110337", "SRR28110339")  # heat shock vs control (Supt16h replicate 2) (negative data is only SRR28110339)
]

paired_metadata_df_gse259306 = create_paired_metadata(
    combined_df=GSE259306_combined, 
    defined_sample_pairs=defined_pairs_gse259306,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse259306 is not None:

    paired_metadata_df_gse259306.write_csv(
        "../../Data/metadata/metadata_paired/GSE259306_modified_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse259306)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_supplementary_file_2,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,supplementary_files_format_and_content,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR28110418""","""SRR28110420""","""rnaseq_abcf2_hs_pos_rep1 vs rnaseq_abcf2_hs_neg_rep1""","""RNAseq_Abcf2_HS_pos_Rep1 vs RNAseq_Abcf2_HS_neg_Rep1""","""GSM8113661 vs GSM8113659""","""Public on Dec 21 2024""","""Feb 26 2024""","""Dec 21 2024""","""SRA""","""1""","""HCT116""","""Homo sapiens""","""9606""","""polyA RNA""","""GPL24676""","""Ali,,Shilatifard""","""ash@northwestern.edu""","""Shilatifard Lab""","""Department of Biochemistry and Molecular Genetics""","""320 E Superior St""","""Chicago""","""IL""","""60611""","""USA""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM8113661 vs GSM8113659""","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8113nnn/GSM8113661/suppl/GSM8113661_RNAseq_Abcf2_HS_pos_Rep1_minus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8113nnn/GSM8113659/suppl/GSM8113659_RNAseq_…","""ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8113nnn/GSM8113661/suppl/GSM8113661_RNAseq_Abcf2_HS_pos_Rep1_plus.bw vs ftp://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8113nnn/GSM8113659/suppl/GSM8113659_RNAseq_A…","""GSE259306""","""0""","""SRX23755224 vs SRX23755222""","""HCT116""","""human colorectal carcinoma cells""","""Abcf2""","""42C 3 hours (heat shock) vs 37C 3 hours (no heat shock)""","""hg38""","""bigwig files for ChIP-seq data""","""SAMN40146632 vs SAMN40146634""","""NEBNext Ultra II Directional RNA Library Prep Kit for Illumina""","""https://doi.org/10.1016/j.molcel.2024.11.007""","""42 C""","""180 min.""","""37 C (inferred)""","""SRR28110418 vs SRR28110420""","""Illumina NovaSeq 6000 sequencing: GSM8113661: RNAseq_Abcf2_HS_pos_Rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM8113659: RNAseq_Abcf2_HS_neg_Rep1 Homo sapiens RNA-Seq""","""TurboCas: A method for locus-specific labeling of genomic regions and isolating their associated protein interactome [RNAseq]""","""SRA1810492""","""cDNA""","""2024-12-25""","""1344113132 vs 1507820708""","""1192645091;1227456195 vs 1345223536;1377706850""","""ILLUMINA""",null,null,"""SRP491877""","""40444594 vs 46336388""","""TurboCas: A method for locus-specific labeling of genomic regions and isolating their associated protein interactome [RNAseq]""",null,"""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 9. GSE247417 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-04-29
- [CLK2 Condensates Reorganize Nuclear Speckles and Induce Intron Retention](https://doi.org/10.1002/advs.202309588)
- GSE271918 と同じ研究グループで、おそらく同じ論文由来のデータ
- 論文には掲載されていないが... (ミス？)
- 42℃, 2h
- コントロールデータも揃っているので解析に使用する予定
- GSE271918 との対応を考えると、おそらく 42 C , 240 min. の推定

In [21]:
GSE247417_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE247417/GSE247417_PEP/GSE247417_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (from supproting information)
    pl.lit("https://doi.org/10.1002/advs.202309588").alias("reference_doi"), # add DOI of the reference paper column (BioRxiv)
    pl.lit("42 C (inferred)").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min. (inferred)").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE247417 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE247417_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE247417_filtered = GSE247417[[s.name for s in GSE247417 if not (s.null_count() == GSE247417.height)]]

GSE247417_combined = GSE247417_PEP.join(
    GSE247417_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE247417_combined = GSE247417_combined[[s.name for s in GSE247417_combined if not (s.null_count() == GSE247417_combined.height)]]

GSE247417_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE247417_modified_metadata.tsv",
    separator='\t'
)

# display(GSE247417_combined)

In [22]:
# Manual defined pairs
defined_pairs_gse247417 = [
    ("SRR26772129", "SRR26772135"), # heat shock vs control
    ("SRR26772128", "SRR26772134"),
    ("SRR26772127", "SRR26772132"),
    ("SRR26772126", "SRR26772131"),
    ("SRR26772125", "SRR26772133"),
    ("SRR26772124", "SRR26772130")
    
]

paired_metadata_df_gse247417 = create_paired_metadata(
    combined_df=GSE247417_combined, 
    defined_sample_pairs=defined_pairs_gse247417,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse247417 is not None:

    paired_metadata_df_gse247417.write_csv(
        "../../Data/metadata/metadata_paired/GSE247417_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse247417)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_type,genotype,treatment,assembly,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR26772129""","""SRR26772135""","""hela_cells_hs-gfp-1 vs hela_cells_hs_control__gfp-1""","""HeLa cells,HS-GFP-1 vs HeLa cells (HS control), GFP-1""","""GSM7889752 vs GSM7889746""","""Public on Nov 09 2023""","""Nov 09 2023""","""Nov 09 2023""","""SRA""","""1""","""HeLa""","""Homo sapiens""","""9606""","""total RNA""","""GPL29480""","""Wei,,Wang""","""wekle@hust.edu.cn""","""13 Hangkong Road""","""Wuhan""","""Hubei""","""430030""","""China""","""DNBSEQ-T7""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7889752 vs GSM7889746""","""NONE""","""GSE247417""","""0""","""SRX22470784 vs SRX22470778""","""HeLa""","""HeLa""","""Control""","""Heat shock vs Untreated""","""Homo sapiens""","""SAMN38188930 vs SAMN38188936""","""unknown""","""https://doi.org/10.1002/advs.202309588""","""42 C (inferred)""","""240 min. (inferred)""","""37 C (inferred)""","""SRR26772129 vs SRR26772135""","""DNBSEQ-T7 sequencing: GSM7889752: HeLa cells,HS-GFP-1 Homo sapiens RNA-Seq vs DNBSEQ-T7 sequencing: GSM7889746: HeLa cells (HS control), GFP-1 Homo sapiens RNA-Seq""","""Effect of CLK2 expression in the regulation of intron retention during heat shock""","""SRA1748434""","""cDNA""","""2023-11-12""","""2600805222 vs 1805009730""","""1624574697;1624015982 vs 1135004324;1134465725""","""DNBSEQ""","""missing""","""Generic""","""HeLa cells,HS-GFP-1 vs HeLa cells (HS control), GFP-1""","""SRP471083""","""28533807 vs 19766799""","""Effect of CLK2 expression in the regulation of intron retention during heat shock""","""HeLa cells,HS-GFP-1 vs HeLa cells (HS control), GFP-1""","""DNBSEQ-T7""","""DNBSEQ-T7 sequencing: GSM7889752: HeLa cells,HS-GFP-1 Homo sapiens RNA-Seq vs DNBSEQ-T7 sequencing: GSM7889746: HeLa cells (HS control), GFP-1 Homo sapiens RNA-Seq""","""HeLa""","""8560142100 vs 5930039700""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7889752 vs GSM7889746""","""GSM7889752_r1 vs GSM7889746_r1""","""Total RNA was isolated using the Trizol Reagent. Three micrograms of RNA were used as input material for the RNA sample preparations. Sequencing libraries were generated according to the following ste…","""missing""","""GSM7889752_r1 vs GSM7889746_r1""","""HeLa""","""GSM7889752 vs GSM7889746""","""2023-11-12""","""PRJNA1037521""","""SRS19484484 vs SRS19484481""","""2023-11-12""","""GSE247417"""
"""SRR2677

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE186004 (PAIRED) NO DOWNLOAD

- validation date: 2025-04-29
- HSF1 のノックアウトとestrageol の関係などに関するデータ
- 熱ストレスに関連するデータがないため、今回は使用しない
- `mcf7_cells_hsf1+_+_control_repeat_1`, `mcf7_cells_hsf1+_+_estrogen_4h_repeat_1`, `mcf7_cells_hsf1-_-_control_repeat_1`, `mcf7_cells_hsf1-_-_estrogen_4h_repeat_1`などのサンプルが存在する

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 10. GSE159802 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [Heat shock factor 1 (HSF1) cooperates with estrogen receptor α (ERα) in the regulation of estrogen action in breast cancer cells](https://doi.org/10.7554/eLife.69843)
- estrogenにさらされているデータは取り除いた状態で解析に使用する

In [23]:
GSE159802_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE159802/GSE159802_PEP/GSE159802_PEP_raw.csv",
    separator=','
).filter(
    ~(pl.col("sample_description").str.contains(r'estrogen')) # remove estrogen data
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column
    pl.lit("https://doi.org/10.7554/eLife.69843").alias("reference_doi"), # add DOI of the reference paper column (eLife)
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE159802 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE159802_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE159802_filtered = GSE159802[[s.name for s in GSE159802 if not (s.null_count() == GSE159802.height)]]

GSE159802_combined = GSE159802_PEP.join(
    GSE159802_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE159802_combined = GSE159802_combined[[s.name for s in GSE159802_combined if not (s.null_count() == GSE159802_combined.height)]]

GSE159802_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE159802_modified_metadata.tsv",
    separator='\t'
)

# display(GSE159802_combined)

In [24]:
# Manual defined pairs
defined_pairs_gse159802 = [
    ("SRR12868789", "SRR12868787"), # heat shock vs control (MCF7 WT)
    ("SRR12868792", "SRR12868790"), # heat shock vs control (MCF7 SCR)
    ("SRR12868795", "SRR12868793"), # heat shock vs control (MCF7 shHSF1)
    ("SRR12868798", "SRR12868796"), # heat shock vs control (MCF7 MIX)
    ("SRR12868801", "SRR12868799"), # heat shock vs control (MCF7 KO#1)
    ("SRR12868804", "SRR12868802")  # heat shock vs control (MCF7 KO#2)  
]

paired_metadata_df_gse159802 = create_paired_metadata(
    combined_df=GSE159802_combined, 
    defined_sample_pairs=defined_pairs_gse159802,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse159802 is not None:

    paired_metadata_df_gse159802.write_csv(
        "../../Data/metadata/metadata_paired/GSE159802_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse159802)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,tumor,sex,age,tissue,genotype,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR12868789""","""SRR12868787""","""a3_mcf7_wt_hs_1h_2h_rec vs a1_mcf7_wt_control""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR12868789 vs SRR12868787""","""SRX9334904 vs SRX9334902""","""A3: MCF7_WT_HS_1h/2h_rec vs A1: MCF7_WT_control""","""GSM4847384 vs GSM4847382""","""Public on Oct 19 2021""","""Oct 21 2020""","""Oct 19 2021""","""SRA""","""1""","""MCF7 cells""","""Homo sapiens""","""9606""","""total RNA""","""MCF7 wild type - hyperhermia (43°C) for 1hour & 2hours of recovery vs MCF7 wild type - untreated""","""GPL18460""","""Tomasz,,Stokowy""","""tomasz.stokowy@uib.no""","""Scientific Computing""","""IT Division""","""University of Bergen""","""Nygardsgaten 5""","""Bergen""","""5020""","""Norway""","""Illumina HiSeq 1500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4847384 vs GSM4847382""","""NONE""","""GSE159802, GSE159803""","""0""","""SRX9334904 vs SRX9334902""","""MCF7""","""adenocarcinoma""","""female""","""69 years""","""breast""","""wild type""","""SAMN16510678 vs SAMN16510680""","""unknown""","""https://doi.org/10.7554/eLife.69843""","""43 C""","""60 min.""","""37 C (inferred)""","""SRR12868789 vs SRR12868787""","""Illumina HiSeq 1500 sequencing: GSM4847384: A3: WT_HS_1h/2h_rec Homo sapiens RNA-Seq vs Illumina HiSeq 1500 sequencing: GSM4847382: A1: WT_control Homo sapiens RNA-Seq""","""Heat Shock Factor 1 (HSF1) supports the ESR1 action in breast cancer (RNA-seq)""","""SRA1146105""","""cDNA""","""2021-10-28""","""1298757403 vs 1226690527""","""941323876;968802645 vs 887573858;919034242""","""ILLUMINA""","""Generic""","""A3: MCF7_WT_HS_1h/2h_rec vs A1: MCF7_WT_control""","""SRP288220""","""21566369 vs 20231317""","""Heat Shock Factor 1 (HSF1) supports the ESR1 action in breast cancer (RNA-seq)""","""A3: MCF7_WT_HS_1h/2h_rec vs A1: MCF7_WT_control""","""Illumina HiSeq 1500""","""Illumina HiSeq 1500 sequencing: GSM4847384: A3: WT_HS_1h/2h_rec Homo sapiens RNA-Seq vs Illumina HiSeq 1500 sequencing: GSM4847382: A1: WT_control Homo sapiens RNA-Seq""","""3278088088 vs 3075160184""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM4847384 vs GSM4847382""","""GSM4847384_r1 vs GSM4847382_r1""","""Total RNA was isolated from all 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 11. GSE73471 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [The bromodomain protein BRD4 regulates splicing during heat shock](https://doi.org/10.1093/nar/gkw729)
- [bromodomain containing 4](https://www.genenames.org/data/gene-symbol-report/#!/hgnc_id/HGNC:13575) の RNA interference (RNAi) を熱ストレスと共に行っている研究
- これも以前の研究で使用済み
- 解析に使用する
- ペア案: `ctrl_1` vs `hs_1`, `hs_2`, `hs_3`
- ペア案: `si_brd4_1` vs `si_brd4_hs_1`, `si_brd4_hs_2`, `si_brd4_hs_3`
- validation date 2: 2025-05-15
- `SRR2522102` のfastpの処理が以下のようなエラーになった

```bash
WARNING: different read numbers of the 209849 pack
Read1 pack size: 256
Read2 pack size: 123

ERROR: input files don't contain identical amount of reads
```

- そのため、今回はこのデータを外す
- (2026-09-29) 論文を見ても、特に stranded RNA-seq と書いていないので、strandness はわからなかった

In [25]:
GSE73471_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE73471/GSE73471_PEP/GSE73471_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq RNA Sample Preparation Kit").alias("rna_library"), # add rna_library column (from supplementary information)
    pl.lit("https://doi.org/10.1093/nar/gkw729").alias("reference_doi"), # add DOI of the reference paper column (NAR)
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("240 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE73471 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE73471_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE73471_filtered = GSE73471[[s.name for s in GSE73471 if not (s.null_count() == GSE73471.height)]]

GSE73471_combined = GSE73471_PEP.join(
    GSE73471_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE73471_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE73471_modified_metadata.tsv",
    separator='\t'
)

# display(GSE73471_combined)

In [26]:
# Manual defined pairs
defined_pairs_gse73471 = [
    ("SRR2522096", "SRR2522093"), # HS 1 vs Ctrl 1
    ("SRR2522098", "SRR2522095"), # si BRD 4 & HS1 vs si BRD4
    ("SRR2522099", "SRR2522093"), # HS 2 vs Ctrl 1
    ("SRR2522101", "SRR2522095"), # si BRD 4 & HS2 vs si BRD4
    # ("SRR2522102", "SRR2522093"), # HS 3 vs Ctrl 1
    ("SRR2522103", "SRR2522095")  # si BRD 4 & HS3 vs si BRD4
]

paired_metadata_df_gse73471 = create_paired_metadata(
    combined_df=GSE73471_combined, 
    defined_sample_pairs=defined_pairs_gse73471,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse73471 is not None:

    paired_metadata_df_gse73471.write_csv(
        "../../Data/metadata/metadata_paired/GSE73471_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse73471)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_treatment_protocol_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,treatment,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR2522096""","""SRR2522093""","""hs_1 vs ctrl_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR2522096 vs SRR2522093""","""SRX1289547 vs SRX1289545""","""HS 1 vs Ctrl 1""","""GSM1895367 vs GSM1895365""","""Public on Sep 07 2016""","""Sep 26 2015""","""May 15 2019""","""SRA""","""1""","""WI38 cell line""","""Homo sapiens""","""9606""","""see Samples section""","""total RNA""","""GPL16791""","""Michal,Ruth,Schweiger""","""mschweig@molgen.mpg.de""","""Schweiger""","""Lehrach""","""Max Planck Institute For Molecular Genetics""","""Ihnestrasse 63-73""","""Berlin""","""14195""","""Germany""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM1895367 vs GSM1895365""","""NONE""","""GSE73471""","""0""","""SRX1289547 vs SRX1289545""","""WI38 lung fibroblast cells""","""To induce heat stress, cells were incubated at 42°C in a waterbath for 4 hours vs none (control)""","""hg19, GRCh37""","""SAMN04115214 vs SAMN04115212""","""TruSeq RNA Sample Preparation Kit""","""https://doi.org/10.1093/nar/gkw729""","""42 C""","""240 min.""","""37 C (inferred)""","""SRR2522096 vs SRR2522093""","""Illumina HiSeq 2500 sequencing: GSM1895367: HS 1 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM1895365: Ctrl 1 Homo sapiens RNA-Seq""","""Homo sapiens""","""SRA301088""","""cDNA""","""2020-04-22 vs 2019-10-28""","""1913286060 vs 1988311996""","""1194492286;1218159232 vs 1236501344;1261260314""","""ILLUMINA""","""Generic""","""HS 1 vs Ctrl 1""","""SRP064222""","""32151112 vs 33226646""","""The bromodomain protein BRD4 regulates splicing during heat shock""","""HS 1 vs Ctrl 1""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM1895367: HS 1 Homo sapiens RNA-Seq vs Illumina HiSeq 2500 sequencing: GSM1895365: Ctrl 1 Homo sapiens RNA-Seq""","""3279413424 vs 3389117892""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM1895367 vs GSM1895365""","""GSM1895367_r1 vs GSM1895365_r1""","""Total RNA was purified using the RNA MicroPrep Kit (Zymo Research). Complementary DNA (cDNA) was generated using SuperScript II Reverse Transcriptase (Invitrogen) according to the manufacturer’s instr…","""GSM1895367 vs GSM1895365""","""GEO""","""WI38 lung fibroblast cells""",""

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 12. GSE53667 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [Heat Shock Alters the Expression of Schizophrenia and Autism Candidate Genes in an Induced Pluripotent Stem Cell Model of the Human Telencephalon](https://doi.org/10.1371/journal.pone.0094968)
- サンプル条件: 39C for 24 hrs
- これも以前の研究で使用済み・解析に使用する
- ペア案: `control_1_he006_` vs `hs_1_he007_`
- ペア案: `control_2_he008_,` vs `hs_2_he009_`

In [27]:
GSE53667_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE53667/GSE53667_PEP/GSE53667_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("unknown").alias("rna_library"), # add rna_library column (Qiagen)
    pl.lit("https://doi.org/10.1371/journal.pone.0094968").alias("reference_doi"), # add DOI of the reference paper column (PLOS ONE)
    pl.lit("39 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("1440 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C").alias("control treatment temperature"), # add control temperature column
    pl.lit("neuronal aggregates (day 50) derived from induced pluripotent stem cells").alias("cell_line"), # Strictly, this is not cell line, but add this as sample
    
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE53667 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE53667_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE53667_filtered = GSE53667[[s.name for s in GSE53667 if not (s.null_count() == GSE53667.height)]]

GSE53667_combined = GSE53667_PEP.join(
    GSE53667_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE53667_combined = GSE53667_combined[[s.name for s in GSE53667_combined if not (s.null_count() == GSE53667_combined.height)]]

GSE53667_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE53667_modified_metadata.tsv",
    separator='\t'
)

# display(GSE53667_combined)

In [28]:
# Manual defined pairs
defined_pairs_gse53667 = [
    ("SRR1061343", "SRR1061341"),
    ("SRR1061344", "SRR1061342")
]

paired_metadata_df_gse53667 = create_paired_metadata(
    combined_df=GSE53667_combined, 
    defined_sample_pairs=defined_pairs_gse53667,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse53667 is not None:

    paired_metadata_df_gse53667.write_csv(
        "../../Data/metadata/metadata_paired/GSE53667_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse53667)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,sample_group,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,cell_line,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR1061343""","""SRR1061341""","""hs_1_he007_ vs control_1_he006_""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR1061343 vs SRR1061341""","""SRX399548 vs SRX399546""","""HS_1 (HE007) vs Control_1 (HE006)""","""GSM1298381 vs GSM1298379""","""Public on Jun 01 2014""","""Dec 26 2013""","""May 15 2019""","""SRA""","""1""","""Homo sapiens""","""9606""","""total RNA""","""GPL11154""","""Herb,,Lachman""","""herb.lachman@einstein.yu.edu""","""Behavioral Genetics""","""Psychiatry""","""Albert Einstein College of Medicine""","""1300 Morris Park Ave., F103""","""Bronx""","""NY""","""10461""","""USA""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM1298381 vs GSM1298379""","""NONE""","""GSE53667""","""0""","""SRX399548 vs SRX399546""","""39C for 24 hrs vs 37C control""","""hg19""","""SAMN02486493 vs SAMN02486491""","""unknown""","""https://doi.org/10.1371/journal.pone.0094968""","""39 C""","""1440 min.""","""37 C""","""neuronal aggregates (day 50) derived from induced pluripotent stem cells""","""SRR1061343 vs SRR1061341""","""Illumina HiSeq 2000 sequencing: GSM1298381: HS_1 (HE007) Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM1298379: Control_1 (HE006) Homo sapiens RNA-Seq""","""Homo sapiens""","""SRA122385""","""cDNA""","""2020-05-21""","""4786644890 vs 4554623413""","""3050142627;3043390459 vs 2896236634;2899678793""","""ILLUMINA""","""Generic""","""HS_1 (HE007) vs Control_1 (HE006)""","""SRP034737""","""32440105 vs 30769108""","""Gene expression profiling in an induced pluripotent stem cell model of the developing human telencephalon: effect of heat shock and its potential impact on the development of neuropsychiatric disorder…","""HS_1 (HE007) vs Control_1 (HE006)""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM1298381: HS_1 (HE007) Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM1298379: Control_1 (HE006) Homo sapiens RNA-Seq""","""6552901210 vs 6215359816""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM1298381 vs GSM1298379""","""GSM1298381_r1 vs GSM1298379_r1""","""total cellular RNA was isolated using the miRNeasy Kit (Qiagen) according to the manufacturer’s protocol RNA libraries were prepared for sequencing using standard Illumina p

&nbsp;

&nbsp;

&nbsp;

&nbsp;

### GSE233555 Ribo-Calibration (PAIRED)

- validation date: 2025-05-02
- [Calibrated ribosome profiling assesses the dynamics of ribosomal flux on transcripts](https://doi.org/10.1038/s41467-024-51258-0)
- 
- `For heat shock treatment, HEK293 cells cultured in 6-well plates were transferred to a water bath set at 43 °C and incubated for 0.5, 1, or 2 h. For recovery experiments, cells were returned to 37 °C and cultured for 0.5, 1, or 4 h.`
- リボソームのプロファイリング手法に関連する研究
- Homo sapiens のRNA-seqのデータを取得
- おそらく `RNA_HS...` で始まるデータが関連しているのでフィルタリング
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に処理を移動

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 13. GSE150922 (PAIRED) 

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [Efficient Heat Shock Response Affects Hyperthermia-Induced Radiosensitization in a Tumor Spheroid Control Probability Assay](https://doi.org/10.3390/cancers13133168)
- 以前の研究でも使用したデータ
- Radiotherapy と Heat stress (hyperthermia) のデータがある
- コントロールとHeat stress のデータを取得する
- Radiotherapy + Heat stress (RT + HT) の複合的なストレスを加えているデータは今回はフィルタリングしておく

In [29]:
GSE150922_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE150922/GSE150922_PEP/GSE150922_PEP_raw.csv",
    separator=','
).filter(
    ((pl.col("sample_title").str.contains(r'_Control_')) | (pl.col("sample_title").str.contains(r'_HT_')))
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("NEBNext Ultra II Directional RNA Library Prep Kit").alias("rna_library"), # add rna_library column from GEO webpage (NEW ENGLAND Biolabs)
    pl.lit("https://doi.org/10.3390/cancers13133168").alias("reference_doi"), # add DOI of the reference paper column (Cancers)
    pl.lit("42.5 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("60 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE150922 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE150922_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE150922_filtered = GSE150922[[s.name for s in GSE150922 if not (s.null_count() == GSE150922.height)]]

GSE150922_combined = GSE150922_PEP.join(
    GSE150922_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE150922_combined = GSE150922_combined[[s.name for s in GSE150922_combined if not (s.null_count() == GSE150922_combined.height)]]

GSE150922_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE150922_modified_metadata.tsv",
    separator='\t'
)

# display(GSE150922_combined)

In [30]:
# Manual defined pairs
defined_pairs_gse150922 = [
    ("SRR11817840", "SRR11817834"),
    ("SRR11817841", "SRR11817835"),
    ("SRR11817842", "SRR11817836"),
    ("SRR11817852", "SRR11817846"),
    ("SRR11817853", "SRR11817847"),
    ("SRR11817854", "SRR11817848")
]

paired_metadata_df_gse150922 = create_paired_metadata(
    combined_df=GSE150922_combined, 
    defined_sample_pairs=defined_pairs_gse150922,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse150922 is not None:

    paired_metadata_df_gse150922.write_csv(
        "../../Data/metadata/metadata_paired/GSE150922_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse150922)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_line,cell_model,treatment,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR11817840""","""SRR11817834""","""fadu_ht_rep1 vs fadu_control_rep1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR11817840 vs SRR11817834""","""SRX8368807 vs SRX8368801""","""FaDu_HT_Rep1 vs FaDu_Control_Rep1""","""GSM4561027 vs GSM4561021""","""Public on Jul 01 2021""","""May 20 2020""","""Jul 01 2021""","""SRA""","""1""","""head-and-neck cancer cell line""","""Homo sapiens""","""9606""","""total RNA""","""GPL24676""","""Oleg,I,Chen""","""oleh.chen@gmail.com""","""OncoRay""","""Handelallee 26""","""Dresden""","""1307""","""Germany""","""Illumina NovaSeq 6000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM4561027 vs GSM4561021""","""NONE""","""GSE150922""","""0""","""SRX8368807 vs SRX8368801""","""FaDu""","""spheroids""","""Hyperthermia vs Control""","""mm9""","""SAMN14981421 vs SAMN14981400""","""NEBNext Ultra II Directional RNA Library Prep Kit""","""https://doi.org/10.3390/cancers13133168""","""42.5 C""","""60 min.""","""37 C (inferred)""","""SRR11817840 vs SRR11817834""","""Illumina NovaSeq 6000 sequencing: GSM4561027: FaDu_HT_Rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM4561021: FaDu_Control_Rep1 Homo sapiens RNA-Seq""","""Next Generation Sequencing of whole SAS and FaDu spheroids transcriptomes treated with Hyperthermia (HT) and Radiation therapy (RT)""","""SRA1077428""","""cDNA""","""2021-07-04""","""1997402221 vs 1925633089""","""1336578785;1410432948 vs 1302808760;1352282453""","""ILLUMINA""","""Generic""","""FaDu_HT_Rep1 vs FaDu_Control_Rep1""","""SRP262461""","""20885613 vs 20191842""","""Next Generation Sequencing of whole SAS and FaDu spheroids transcriptomes treated with Hyperthermia (HT) and Radiation therapy (RT)""","""FaDu_HT_Rep1 vs FaDu_Control_Rep1""","""Illumina NovaSeq 6000""","""Illumina NovaSeq 6000 sequencing: GSM4561027: FaDu_HT_Rep1 Homo sapiens RNA-Seq vs Illumina NovaSeq 6000 sequencing: GSM4561021: FaDu_Control_Rep1 Homo sapiens RNA-Seq""","""5771729167 vs 5633598867""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM4561027 vs GSM4561021""","""GSM4561027_r1 vs GSM4561021_r1""","""Total cellular RNA was isolated from SAS and FaDu spheroids using the RNeasy Mini kit (QIAGEN, Germany) according to the manufacturer's protocol. RNA sequencing library preparations used the NEBNext U…","""GSM45

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 14. GSE189578 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- おそらく以下の論文がこのデータの元になっている
- [Hyperthermia inhibits growth of nasopharyngeal carcinoma through degradation of c-Myc](https://doi.org/10.1080/02656736.2022.2038282)
- `human nasopharyngeal carcinoma cell lines` のCNE2 lineのデータに対して 43℃の熱曝露、およびおそらく30分間 (ここははっきりわからない...) を行っているデータ
- コントロール、熱曝露のデータが揃っているため、このデータは使用できると考え収集

In [31]:
GSE189578_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE189578/GSE189578_PEP/GSE189578_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("TruSeq RNA Sample Preparation Kit").alias("rna_library"), # add rna_library column from GEO webpage (Illumina)
    pl.lit("https://doi.org/10.1080/02656736.2022.2038282").alias("reference_doi"), # add DOI of the reference paper column (International Journal of Hyperthermia)
    pl.lit("43 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("30 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE189578 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE189578_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE189578_filtered = GSE189578[[s.name for s in GSE189578 if not (s.null_count() == GSE189578.height)]]

GSE189578_combined = GSE189578_PEP.join(
    GSE189578_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE189578_combined = GSE189578_combined[[s.name for s in GSE189578_combined if not (s.null_count() == GSE189578_combined.height)]]

GSE189578_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE189578_modified_metadata.tsv",
    separator='\t'
)

# display(GSE189578_combined)

In [32]:
# Manual defined pairs
defined_pairs_gse189578 = [
    ("SRR17038864", "SRR17038861"),
    ("SRR17038865", "SRR17038862"),
    ("SRR17038866", "SRR17038863")
]

paired_metadata_df_gse189578 = create_paired_metadata(
    combined_df=GSE189578_combined, 
    defined_sample_pairs=defined_pairs_gse189578,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse189578 is not None:

    paired_metadata_df_gse189578.write_csv(
        "../../Data/metadata/metadata_paired/GSE189578_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse189578)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,cell_type,cell_line,treatment,genome_build,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR17038864""","""SRR17038861""","""cne2_ht43c_1 vs cne2_ctrl_1""","""cDNA""","""Homo sapiens""","""PAIRED""","""SRA""","""SRR17038864 vs SRR17038861""","""SRX13228664 vs SRX13228661""","""CNE2 HT43C 1 vs CNE2 Ctrl 1""","""GSM5704146 vs GSM5704143""","""Public on Jun 01 2022""","""Nov 25 2021""","""Jun 01 2022""","""SRA""","""1""","""Nasopharyngeal carcinoma""","""Homo sapiens""","""9606""","""polyA RNA""","""C2_H1 vs C2_NC1""","""GPL11154""","""le,xiao,li""","""lixiaole929929@yeah.net""","""the First Affiliated Hospital of Zhengzhou""","""wulibao street""","""zhengzhou""","""450000""","""China""","""Illumina HiSeq 2000""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM5704146 vs GSM5704143""","""NONE""","""GSE189578""","""0""","""SRX13228664 vs SRX13228661""","""Nasopharyngeal epithelium-derived cells""","""CNE2""","""heat vs control""","""GRCh38.91""","""SAMN23441146 vs SAMN23441143""","""TruSeq RNA Sample Preparation Kit""","""https://doi.org/10.1080/02656736.2022.2038282""","""43 C""","""30 min.""","""37 C (inferred)""","""SRR17038864 vs SRR17038861""","""Illumina HiSeq 2000 sequencing: GSM5704146: CNE2 HT43℃ 1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM5704143: CNE2 Ctrl 1 Homo sapiens RNA-Seq""","""The effect of hyperthermia on nasopharyngeal carcinoma cells""","""SRA1334989""","""cDNA""","""2022-06-21""","""1925253484 vs 1981049513""","""1417539832;1448914861 vs 1465949872;1491314395""","""ILLUMINA""","""Generic""","""CNE2 HT43C 1 vs CNE2 Ctrl 1""","""SRP347901""","""19847103 vs 20537600""","""The effect of hyperthermia on nasopharyngeal carcinoma cells""","""CNE2 HT43C 1 vs CNE2 Ctrl 1""","""Illumina HiSeq 2000""","""Illumina HiSeq 2000 sequencing: GSM5704146: CNE2 HT43℃ 1 Homo sapiens RNA-Seq vs Illumina HiSeq 2000 sequencing: GSM5704143: CNE2 Ctrl 1 Homo sapiens RNA-Seq""","""Nasopharyngeal epithelium-derived cells""","""5954130900 vs 6161280000""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM5704146 vs GSM5704143""","""GSM5704146_r1 vs GSM5704143_r1""","""Total RNA was extracted with AMBION AM1560 kit. Following ""TruSeq RNA Sample Preparation Guide"".""","""GSM5704146 vs GSM5704143""","""GEO""","""CNE2""","""2022-06-20""","""PRJNA783644""","""SRS11153675 vs SRS11153672""","""2022-0

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 15. GSE227393 (PAIRED)

- ✅ DOWNLOAD
- ✅ TRIMMING
- ✅ QUANTIFICATION

&nbsp;

- validation date: 2025-05-02
- [Comprehensive multi-omics analysis reveals WEE1 as a synergistic lethal target with hyperthermia through CDK1 super-activation](https://doi.org/10.1038/s41467-024-46358-w)
- OVCAR8 cells に対して hyperthemia (42℃) for 90 mins を曝露しているデータ (Ctrl は 37℃)
- しっかりペアのデータが存在しているので収集

In [33]:
GSE227393_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE227393/GSE227393_PEP/GSE227393_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    pl.lit("VAHTS mRNA-seq v2 Library Prep Kit for Illumina").alias("rna_library"), # add rna_library column from GEO webpage.
    pl.lit("https://doi.org/10.1038/s41467-024-46358-w").alias("reference_doi"), # add DOI of the reference paper column (Nature Communications)
    pl.lit("42 C").alias("stress treatment temperature"), # add heat treatment temperature column
    pl.lit("90 min.").alias("stress treatment time"), # add heat treatment time column (minutes)
    pl.lit("37 C (inferred)").alias("control treatment temperature") # add control temperature column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# metadata from iseq
GSE227393 = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE227393_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE227393_filtered = GSE227393[[s.name for s in GSE227393 if not (s.null_count() == GSE227393.height)]]

GSE227393_combined = GSE227393_PEP.join(
    GSE227393_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE227393_combined = GSE227393_combined[[s.name for s in GSE227393_combined if not (s.null_count() == GSE227393_combined.height)]]

GSE227393_combined.write_csv(
    "../../Data/metadata/metadata_modified/GSE227393_modified_metadata.tsv",
    separator='\t'
)

# display(GSE227393_combined)

In [34]:
# Manual defined pairs
defined_pairs_gse227393 = [
    ("SRR23868571", "SRR23868570"),
    ("SRR24982378", "SRR24982380"),
    ("SRR24982377", "SRR24982379")
]

paired_metadata_df_gse227393 = create_paired_metadata(
    combined_df=GSE227393_combined, 
    defined_sample_pairs=defined_pairs_gse227393,
    sample_id_column_name="accession" # default value, but specified to clarify
)

if paired_metadata_df_gse227393 is not None:

    paired_metadata_df_gse227393.write_csv(
        "../../Data/metadata/metadata_paired/GSE227393_paired_metadata.tsv",
        separator='\t'
    )

    display(paired_metadata_df_gse227393)

else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_platform_id,sample_contact_name,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,tissue,cell_line,cell_type,treatment,sample_accession,rna_library,reference_doi,stress treatment temperature,stress treatment time,control treatment temperature,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,country,ncbi_reporting_standard,sample_description,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,cell_type_right,base_count,library_strategy,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,collection_date,experiment_alias,cell_line_right,library_name,first_created,study_accession,secondary_sample_accession,first_public,study_alias,tissue_type
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR23868571""","""SRR23868570""","""ovcar8_ht_1 vs ovcar8_ctrl_1""","""OVCAR8_HT_1 vs OVCAR8_Ctrl_1""","""GSM7099898 vs GSM7099897""","""Public on Dec 12 2023""","""Mar 15 2023""","""Dec 12 2023""","""SRA""","""1""","""Ovary""","""Homo sapiens""","""9606""","""total RNA""",""">2 × 10^6 cells were collected and resolved in TRIzol (Takara, 9109) for RNA extraction, RNA was harvested using QIAGEN’s AllPrep RNA Mini Kit. 2ug of total RNA was used for the construction of sequen…","""GPL28038""","""Hu,,Xingyuan""","""同济医院""","""湖北武汉""","""武汉""","""Hubei""","""430000""","""USA""","""DNBSEQ-G400""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM7099898 vs GSM7099897""","""NONE""","""GSE227393""","""0""","""SRX19680623 vs SRX19680624""","""Ovary""","""OVCAR8""","""ovarian cancer cell""","""hyperthermia stimulation vs control""","""SAMN33768760 vs SAMN33768761""","""VAHTS mRNA-seq v2 Library Prep Kit for Illumina""","""https://doi.org/10.1038/s41467-024-46358-w""","""42 C""","""90 min.""","""37 C (inferred)""","""SRR23868571 vs SRR23868570""","""DNBSEQ-G400 sequencing: GSM7099898: OVCAR8_HT_1 Homo sapiens RNA-Seq vs DNBSEQ-G400 sequencing: GSM7099897: OVCAR8_Ctrl_1 Homo sapiens RNA-Seq""","""Comprehensive multi-omics analysis revealed WEE1 as a synergistic lethal target with hyperthermia through CDK super-activation""","""SRA1604868""","""cDNA""","""2023-12-15""","""4394601566 vs 4421750848""","""2669114392;2784359530 vs 2673517141;2823317431""","""DNBSEQ""","""missing""","""Generic""","""OVCAR8_HT_1 vs OVCAR8_Ctrl_1""","""SRP427401""","""22185197 vs 22206309""","""Comprehensive multi-omics analysis revealed WEE1 as a synergistic lethal target with hyperthermia through CDK super-activation""","""OVCAR8_HT_1 vs OVCAR8_Ctrl_1""","""DNBSEQ-G400""","""DNBSEQ-G400 sequencing: GSM7099898: OVCAR8_HT_1 Homo sapiens RNA-Seq vs DNBSEQ-G400 sequencing: GSM7099897: OVCAR8_Ctrl_1 Homo sapiens RNA-Seq""","""ovarian cancer cell""","""6655559100 vs 6661892700""","""RNA-Seq""","""TRANSCRIPTOMIC""","""PAIRED""","""public""","""GSM7099898 vs GSM7099897""","""GSM7099898_r1 vs GSM7099897_r1""",""">2 × 10^6 cells were collected and resolved in TRIzol (Takara, 9109) for RNA extraction RNA was harvested using QIAGEN's AllPrep RNA Mini Kit. 2ug of total RNA was used for the construction of 

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Concat dataframe (RNA-seq only)

In [35]:
dfs_rna_seq_all: list[pl.DataFrame | None] = [
        # Single-end
        paired_metadata_df_gse253874,
        paired_metadata_df_gse235351,
        paired_metadata_df_gse124609_rna_seq,
        paired_metadata_df_gse115973,
        paired_metadata_df_gse79680,

        # Paired-end
        paired_metadata_df_gse267930,
        paired_metadata_df_gse271918,
        paired_metadata_df_gse259306,
        paired_metadata_df_gse247417,
        paired_metadata_df_gse159802,
        paired_metadata_df_gse73471,
        paired_metadata_df_gse53667,
        paired_metadata_df_gse150922,
        paired_metadata_df_gse189578,
        paired_metadata_df_gse227393
]

dfs_rna_seq:  list[pl.DataFrame] = [df for df in dfs_rna_seq_all if df is not None]

In [36]:
rna_seq_true_control = pl.concat(dfs_rna_seq, how="diagonal")

rna_seq_true_control.write_csv(
    f"../../Data/metadata/metadata_{VERSION}/rna_seq_true_control_metadata_{execute_date}.tsv",
    separator='\t'
)